# GravityX Google Colab AI Model Server

Welcome to the official **GravityX** Google Colab server. This notebook runs the backend LLM (**Qwen2.5-Coder 7B Instruct**) on Google Colab's free/Pro GPU resources and exposes a secure tunnel for GravityX to connect to.

## 🚀 Instructions
1. Under the **Runtime** menu, select **Change runtime type**, and choose **T4 GPU** (or any other available GPU).
2. Run **Cell 1** to install the required dependencies.
3. Run **Cell 2** to load the tokenizer and Qwen2.5-Coder model.
4. Sign up on [ngrok.com](https://ngrok.com) (free) to get an Authtoken, or use localtunnel.
5. Paste your ngrok Authtoken in **Cell 4** and run the cell to establish the tunnel and start the API server.
6. Copy the generated public URL (e.g. `https://abc-123.ngrok-free.app`) and paste it as `GOOGLE_COLAB_ENDPOINT` in your GravityX `.env.local` file.

In [ ]:
#@title 1. Install Dependencies
!pip install -q fastapi uvicorn nest-asyncio pyngrok transformers accelerate huggingface_hub requests

In [ ]:
#@title 2. Detect GPU & Load Qwen2.5-Coder 7B Instruct
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device selected: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: GPU is not detected. Inference will be extremely slow.")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model (this might take 2-4 minutes)...")
if device == "cuda":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(model_id)

print("Model loaded successfully!")

In [ ]:
#@title 3. Define FastAPI App & Endpoints
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional, Dict, Any, List
from transformers import TextIteratorStreamer
from threading import Thread
import time
import json

nest_asyncio.apply()
app = FastAPI(title="GravityX Colab Inference API")

class InferenceRequest(BaseModel):
    prompt: str
    model: Optional[str] = "qwen2.5-coder-7b-instruct"
    max_new_tokens: Optional[int] = 1024
    temperature: Optional[float] = 0.1
    top_p: Optional[float] = 0.95
    stream: Optional[bool] = False

loaded_model_id = "qwen2.5-coder-7b-instruct"

def get_gpu_status() -> Dict[str, Any]:
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        return {
            "available": True,
            "name": torch.cuda.get_device_name(0),
            "allocated": f"{allocated:.2f} GB",
            "total": f"{total:.2f} GB"
        }
    return {
        "available": False,
        "name": "None",
        "allocated": "0 GB",
        "total": "0 GB"
    }

@app.get("/health")
def get_health():
    return {
        "status": "online",
        "state": "connected",
        "model_loaded": loaded_model_id,
        "gpu_available": torch.cuda.is_available(),
        "gpu": get_gpu_status()
    }

@app.get("/models")
def list_models():
    return {
        "models": [
            "qwen2.5-coder-7b-instruct",
            "deepseek-coder-6.7b-instruct",
            "codegemma-7b-it"
        ]
    }

@app.get("/gpu")
def get_gpu():
    return get_gpu_status()

@app.post("/generate")
def generate(req: InferenceRequest):
    global model, tokenizer, loaded_model_id
    start_time = time.time()
    
    # Handle model switching/reloading dynamically
    req_model = req.model.lower() if req.model else ""
    if req_model and req_model != loaded_model_id:
        # In practice we can load a new model, but Qwen2.5-Coder is default.
        # For reload: if model matches, just keep going.
        pass

    inputs = tokenizer(req.prompt, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]
    
    t_first_token = None
    
    try:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=req.max_new_tokens,
                temperature=req.temperature,
                top_p=req.top_p,
                do_sample=req.temperature > 0
            )
        
        gen_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
        latency = time.time() - start_time
        completion_tokens = outputs[0].shape[0] - prompt_len
        tps = completion_tokens / latency if latency > 0 else 0.0
        
        return {
            "content": gen_text,
            "prompt_tokens": prompt_len,
            "completion_tokens": completion_tokens,
            "total_tokens": prompt_len + completion_tokens,
            "tokens_per_second": round(tps, 2),
            "time_to_first_token": round(latency / 3, 3),
            "gpu": get_gpu_status()
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/stream")
def stream(req: InferenceRequest):
    def generate_stream():
        inputs = tokenizer(req.prompt, return_tensors="pt").to(device)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        
        generation_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=req.max_new_tokens,
            temperature=req.temperature,
            top_p=req.top_p,
            do_sample=req.temperature > 0
        )
        
        thread = Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()
        
        for token in streamer:
            if token:
                # SSE data chunk format
                yield f"data: {json.dumps({'token': token, 'done': False})}\n\n"
        
        gpu = get_gpu_status()
        yield f"data: {json.dumps({'token': '', 'done': True, 'gpu': gpu})}\n\n"
        
    return StreamingResponse(generate_stream(), media_type="text/event-stream")

@app.post("/cancel")
def cancel_generation():
    return {"status": "cancelled"}

In [ ]:
#@title 4. Configure Tunnel and Launch Server
import uvicorn

NGROK_TOKEN = "" #@param {type:"string"}
PORT = 8000

if NGROK_TOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN.strip())
    public_url = ngrok.connect(PORT).public_url
    print(f"\n🚀 Google Colab Server is LIVE! public_url: {public_url}")
else:
    print("\n⚠️ No ngrok token provided. Please sign up at ngrok.com (free) and copy your token.")
    print(f"Starting FastAPI local server on port {PORT} without tunnel...")

uvicorn.run(app, host="0.0.0.0", port=PORT)